# 🧠 CalRetail — Voice of Customer Mining
## Goal
Mine review text aspects and sentiment frequencies, including a real month-by-month rating trend.

## Algorithmic Explanation
**Aspect-Based Sentiment Keyword Miner**
1. Read reviewer write-ups from customer_reviews.
2. Extract aspect keywords (e.g. price, service, quality, delivery).
3. Map sentiment scores directly from customer ratings.
4. Pass through each review's real `review_date` so downstream monthly trends are computed from
   the actual review timeline, not fabricated placeholder months.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")


In [ ]:
revs = load_table('customer_reviews')
revs['review_text'] = revs['review_text'].fillna('')

aspects = {
    'quality': ['quality', 'durable', 'material', 'fabric'],
    'delivery': ['delivery', 'shipping', 'late', 'fast'],
    'price': ['price', 'value', 'cheap', 'cost']
}
print(f"Voice of customer miner ready on {len(revs)} reviews.")


In [ ]:
def mine_customer_reviews(product_id=None, date_from=None, date_to=None):
    global revs, aspects
    df_revs = revs.copy()
    if product_id:
        df_revs = df_revs[df_revs['product_id'] == product_id]
    if date_from:
        df_revs = df_revs[df_revs['review_date'] >= date_from]
    if date_to:
        df_revs = df_revs[df_revs['review_date'] <= date_to]
        
    results = []
    for idx, row in df_revs.head(1000).iterrows(): # subset for speed
        txt = str(row['review_text']).lower()
        rating = row['rating']
        
        # map aspect
        detected_asp = "General"
        for asp, keywords in aspects.items():
            if any(k in txt for k in keywords):
                detected_asp = asp
                break
                
        # sentiment logic
        sentiment = "Neutral"
        if rating >= 4: sentiment = "Positive"
        elif rating <= 2: sentiment = "Negative"
        
        results.append({
            "review_id": row['review_id'],
            "review_date": row['review_date'],
            "aspect": detected_asp,
            "sentiment": sentiment,
            "rating": rating
        })
    return results

mined_data = mine_customer_reviews()
print("VoC Output Sample:", json.dumps(mined_data[0] if mined_data else {}, indent=2))

In [ ]:
print("=== CALRETAIL VOICE OF CUSTOMER BOARD ===")
voc_df = pd.DataFrame(mined_data)
# aspect breakdown
print("Reviews Count per Aspect Category:")
print(voc_df['aspect'].value_counts())
print("\nSentiment Profile Share:")
print(voc_df.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0))
